# ¿Se puede ser robusto sin confiar en el modelo?

Hasta ahora usamos el modelo para *ayudar* al control.  
Pero… ¿qué pasa cuando el modelo es malo, incompleto o directamente incierto?

En esta notebook exploramos **robustez**:  
la capacidad de mantener un comportamiento aceptable **a pesar** de errores de modelo y perturbaciones externas.

---

## Ensayo propuesto

Compararemos dos estrategias:

- **Control PD**
- **Control por Modos Deslizantes (Sliding Modes)**

sobre una misma planta, sometida a:

- carga adicional  
- errores en los parámetros dinámicos  
- perturbaciones externas  

La referencia será simple, para que el foco esté en el **comportamiento del control**, no en la trayectoria.

---

## Preguntas para guiar la observación

- ¿Cuál de los controles mantiene mejor el tracking cuando el modelo falla?
- ¿Qué ocurre con el esfuerzo de control?
- ¿Aparece *chattering*? ¿Por qué?
- ¿Cómo responde cada control al ruido de medición?

---

## Objetivo de la notebook

Entender que:

- **Robustez no es sinónimo de suavidad**
- El control discontinuo puede ser una **elección de diseño**, no un defecto
- Confiar menos en el modelo suele implicar pagar un precio en el actuador

El objetivo no es “elegir el mejor control”, sino **entender el compromiso**.


In [1]:
import math
import numpy as np
#from scipy.integrate import solve_ivp
import roboticstoolbox as rtb         # la funcionalidad específica de robótica está en el toolbox
import spatialmath as sm              # spatialmath tiene define los grupos de transformaciones especiales: rotación SO3 y eculideo SE3
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
# Preparo el modelo de un doble péndulo
dp = rtb.DHRobot(
    [
        rtb.RevoluteDH(a=200e-3,m=1.5,
                r=np.array([-100, 0, 0]) * 1e-3,
                I=np.array([0,0,0,0,0,0,0,0,1e-3]),
                B=0.1, G=1),
        rtb.RevoluteDH(a=200e-3,m=1,
                r=np.array([-100, 0, 0]) * 1e-3,
                I=np.array([0,0,0,0,0,0,0,0,1e-3]),
                B=0.1, G=1),
    ],
    gravity = np.array([0, -9.8, 0]), # Ojo con el signo, la gravedad va hacia abajo con signo positivo
    name = "DoblePendulo")

qr = np.array([-np.pi/2,0])
qz = np.zeros((2,))

print(dp)
print(dp.dynamics())

DHRobot: DoblePendulo, 2 joints (RR), dynamics, standard DH parameters
┌─────┬────┬─────┬──────┐
│ θⱼ  │ dⱼ │ aⱼ  │  ⍺ⱼ  │
├─────┼────┼─────┼──────┤
│  q1 │  0 │ 0.2 │ 0.0° │
│  q2 │  0 │ 0.2 │ 0.0° │
└─────┴────┴─────┴──────┘

┌──┬──┐
└──┴──┘

┌───────┬──────┬──────────────┬────────────────────────────┬────┬──────┬────────┬────┐
│   j   │  m   │      r       │             I              │ Jm │  B   │   Tc   │ G  │
├───────┼──────┼──────────────┼────────────────────────────┼────┼──────┼────────┼────┤
│ link1 │  1.5 │ -0.1,  0,  0 │  0,  0,  0.001,  0,  0,  0 │  0 │  0.1 │  0,  0 │  1 │
│ link2 │  1   │ -0.1,  0,  0 │  0,  0,  0.001,  0,  0,  0 │  0 │  0.1 │  0,  0 │  1 │
└───────┴──────┴──────────────┴────────────────────────────┴────┴──────┴────────┴────┘

None


## SMC: Sliding Mode Controller

Desarrollamos el control por modos deslizantes a partir de una ley un controlador de estructura variable del estilo:

$$ u = \begin{cases}  U \quad s<0 \\ -U \quad s>0 \end{cases} $$

El objetivo del control es llevar al sistema hacia una superficie en el espacio de fases donde pueda deslizarse hacia el target.

La superficie de deslizamiento se define de manera que cada eje evolucione hacia el origen con una dinámica de primer orden.


In [ ]:
Tfin = 1
Ts = 1E-3
t = np.arange(0,Tfin,Ts)
Ns = len(t)

u = np.zeros((Ns,dp.n))
s = np.zeros_like(u)
q = np.zeros((Ns,dp.n))
qd = np.zeros((Ns,dp.n))
q[0] = np.r_[0,-np.pi/2]
qd_f = np.copy(qd)

# Ley de control SMC
def smc_controller(robot, t, q, qd,
                    Lambda=10*np.diag([6,6]), K=1*np.diag([15,7]), phi=1):

    s  = qd + Lambda @ q

    M = robot.inertia(q)
    C = robot.coriolis(q, qd)
    G = robot.gravload(q)

    # Control equivalente (inverse dynamics)
    tau_eq = M @ (qdd_des - Lambda @ ed) + C @ (qd_des - Lambda @ e) + G

    # Término de conmutación suavizado (tanh)
    tau_sw = K @ np.tanh(s / phi)

    tau = tau_eq - tau_sw
    return tau,s

# Ley de control PD
def pd_controller(robot,t,q,qd):
  Kp = 100
  Kd = 5
  u = Kp * (q_des-q) + Kd * (-qd)
  return u

# Filtro de velocidad
ws = 2*np.pi/Ts
wf = ws/10
alpha = np.exp(-wf*Ts)

# Simulación
for idx in tqdm(range(1,Ns)):
  u[idx],s[idx] = smc_controller(dp,t[idx],q[idx-1],qd_f[idx-1])

  tg = dp.nofriction(coulomb=True,viscous=False).fdyn(Ts,q[idx-1],Q=lambda *args, **kwargs: u[idx],qd0=qd[idx-1],progress=False)

  q[idx] = tg.q[-1]
  qd[idx] = tg.qd[-1]
  qd_f[idx] = alpha * qd_f[idx-1] + (1-alpha) * qd[idx]

plt.figure()
plt.plot(t,q*180/np.pi,lw=2)
plt.xlabel('Tiempo (s)')
plt.ylabel('Angulo (deg)')
plt.figure()
plt.plot(t,u)
plt.ylim((-10,10))
plt.xlabel('Tiempo (s)')
plt.ylabel('Torque (Nm)')

plt.figure()
plt.plot(t,s)
plt.xlabel('Tiempo (s)')
plt.ylabel('S (rad)')
plt.show()
